# Topic 1 practical: investigate what the summaries miss

This is the participant workspace. The framing and helper functions are supplied, but you choose comparison thresholds, record predictions before seeing results, and defend an interpretation.

Work through **observe → predict → implement → compare → interpret**. There is no single “best” threshold to discover.

**Use another data set.** Supply a table with columns `dataset`, `x` and `y`, then rerun the plotting and summary cells. A single point cloud can be given one dataset label.

**◇ Object check.** A point cloud is a finite representation. A threshold graph is constructed from that point cloud, a metric and a threshold. Neither is automatically the latent scientific object.



## Setup: supplied tools

Run this cell first. Read the docstrings, but do not worry about reproducing the helper functions. Your task is to use them deliberately.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(3024)

def standardise_shape(points):
    """Centre and whiten a two-dimensional point cloud."""
    centred = np.asarray(points, dtype=float) - np.mean(points, axis=0)
    covariance = np.cov(centred, rowvar=False)
    values, vectors = np.linalg.eigh(covariance)
    return centred @ vectors @ np.diag(values ** -0.5)

def scalar_summary(points):
    return {
        'mean_x': np.mean(points[:, 0]),
        'mean_y': np.mean(points[:, 1]),
        'sd_x': np.std(points[:, 0], ddof=1),
        'sd_y': np.std(points[:, 1], ddof=1),
        'correlation': np.corrcoef(points.T)[0, 1],
    }

def pairwise_distances(points):
    differences = points[:, None, :] - points[None, :, :]
    return np.sqrt(np.sum(differences ** 2, axis=2))

def threshold_graph_statistics(points, epsilon):
    """Return V, E, components and graph cycle rank at one threshold."""
    distances = pairwise_distances(points)
    adjacency = (distances <= epsilon) & (distances > 0)
    vertices = len(points)
    edges = int(np.sum(adjacency) // 2)
    unseen = set(range(vertices))
    components = 0
    while unseen:
        components += 1
        stack = [unseen.pop()]
        while stack:
            vertex = stack.pop()
            neighbours = set(np.flatnonzero(adjacency[vertex])) & unseen
            unseen -= neighbours
            stack.extend(neighbours)
    cycle_rank = edges - vertices + components
    return {'vertices': vertices, 'edges': edges,
            'components': components, 'graph_cycle_rank': cycle_rank}

print('Environment ready. Random seed: 3024')

## 1. Observe: the canonical Datasaurus Dozen

The included CSV contains all 1,846 observations from the 13 canonical datasets. Each group has 142 points. Before plotting, predict how closely their coordinate means, standard deviations and correlations will agree.

The loader uses a local course copy when available and otherwise reads the public course dataset directly.

In [ ]:
from pathlib import Path
import pandas as pd

def load_datasaurus():
    candidates = [
        Path('datasaurus_dozen.csv'),
        Path('data/datasaurus_dozen.csv'),
        Path('../../data/datasaurus_dozen.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate)
    url = 'https://shannondeealgar.github.io/tda-masterclass/data/datasaurus_dozen.csv'
    return pd.read_csv(url)

datasaurus = load_datasaurus()
dataset_order = list(datasaurus['dataset'].drop_duplicates())

groups = datasaurus.groupby('dataset', sort=False)
summaries = groups.agg(
    n=('x', 'size'),
    mean_x=('x', 'mean'),
    mean_y=('y', 'mean'),
    sd_x=('x', 'std'),
    sd_y=('y', 'std'),
)
summaries['correlation'] = [group['x'].corr(group['y']) for _, group in groups]
print(summaries.round(3).to_string())

fig, axes = plt.subplots(4, 4, figsize=(10, 10), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, name in zip(axes.flat, dataset_order):
    group = datasaurus[datasaurus['dataset'] == name]
    ax.scatter(group['x'], group['y'], s=10)
    ax.set_title(name.replace('_', ' '))
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
for ax in axes.flat[len(dataset_order):]:
    ax.axis('off')
fig.supxlabel('x'); fig.supylabel('y')
plt.show()

# Retain one canonical circle for the controlled transformation section.
ring_raw = datasaurus.loc[datasaurus['dataset'] == 'circle', ['x', 'y']].to_numpy()
ring = standardise_shape(ring_raw)


### Unpack what your eyes call obvious

Choose two datasets that look clearly different. Before calculating anything, record the evidence your visual system seems to use.

| Visual cue | What do you notice? | Which dataset shows it most clearly? |
|---|---|---|
| Local density or crowding |  |  |
| Nearest-neighbour spacing |  |  |
| Direction or elongation |  |  |
| Boundary or curvature |  |  |
| Clusters, branches or empty regions |  |  |
| Symmetry or repetition |  |  |
| Local versus global organisation |  |  |

Now predict which of these geometric descriptors will best separate your pair: median nearest-neighbour distance, convex-hull area, radial variation or covariance anisotropy. Also predict one important feature that all four descriptors will miss.

In [ ]:
def convex_hull(points):
    """Return convex-hull vertices in counter-clockwise order."""
    ordered = sorted(set(map(tuple, points)))
    if len(ordered) <= 1:
        return np.asarray(ordered)
    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])
    lower = []
    for point in ordered:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], point) <= 0:
            lower.pop()
        lower.append(point)
    upper = []
    for point in reversed(ordered):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], point) <= 0:
            upper.pop()
        upper.append(point)
    return np.asarray(lower[:-1] + upper[:-1])

def geometric_descriptors(points):
    distances = pairwise_distances(points)
    np.fill_diagonal(distances, np.inf)
    nearest = distances.min(axis=1)
    hull = convex_hull(points)
    hull_area = 0.5 * abs(np.dot(hull[:, 0], np.roll(hull[:, 1], 1))
                          - np.dot(hull[:, 1], np.roll(hull[:, 0], 1)))
    radii = np.linalg.norm(points - points.mean(axis=0), axis=1)
    eigenvalues = np.linalg.eigvalsh(np.cov(points.T))
    return {
        'median nearest distance': np.median(nearest),
        'convex-hull area': hull_area,
        'radial coefficient of variation': radii.std() / radii.mean(),
        'covariance anisotropy': eigenvalues[-1] / eigenvalues[0],
    }

chosen_names = ['dino', 'circle']  # TODO: choose any two dataset names
comparison = {}
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), constrained_layout=True)
for ax, name in zip(axes, chosen_names):
    points = datasaurus.loc[datasaurus['dataset'] == name, ['x', 'y']].to_numpy()
    comparison[name] = geometric_descriptors(points)
    hull = convex_hull(points)
    closed_hull = np.vstack((hull, hull[0]))
    ax.scatter(points[:, 0], points[:, 1], s=12, label='observations')
    ax.plot(closed_hull[:, 0], closed_hull[:, 1], color='tab:orange',
            linewidth=1.5, label='convex hull')
    ax.set(title=name, xlim=(0, 100), ylim=(0, 100), aspect='equal')
axes[0].legend(loc='lower left', fontsize=8)
plt.show()
display(pd.DataFrame(comparison).T.round(3))


### Compare the geometric descriptors before moving on

Which predictions were supported? Explain what each of the four geometric descriptors measured and identify visible organisation it missed. Convex-hull area, for example, cannot detect an internal gap. Nearest-neighbour spacing, convex-hull area, radial variation and covariance anisotropy ask richer geometric questions than coordinate means, coordinate standard deviations and Pearson correlation, but the four descriptors do not provide a complete account of shape.

Now make a **topological preview without computing topology**. Choose four contrasting datasets and complete this worksheet.

| Dataset | Apparent pieces at a local scale | Candidate loop-like gaps | What might change as neighbourhoods expand? | Modelling caution |
|---|---|---|---|---|
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |
|  |  |  |  |  |

Use provisional language: “appears”, “suggests” and “might persist”. A visible gap is not yet a homology class. Topology asks about connection and holes after a space has been constructed; persistent topology asks how those features change across scale.

## 2. Predict: transformations

Complete the transformation prediction table before running the next code cell.

| Transformation | Continuous ring still homeomorphic to a circle? | Pairwise Euclidean distances unchanged? | Fixed-threshold graph necessarily unchanged? |
|---|---|---|---|
| Translation |  |  |  |
| Rotation |  |  |  |
| Anisotropic stretch |  |  |  |

Now run the comparison and explain any prediction you would revise.

In [ ]:
angle = np.deg2rad(35)
rotation = np.array([[np.cos(angle), -np.sin(angle)],
                     [np.sin(angle),  np.cos(angle)]])
transformed = {
    'original': ring,
    'translated': ring + np.array([3.0, -1.5]),
    'rotated': ring @ rotation.T,
    'anisotropically stretched': ring @ np.diag([1.8, 0.55]),
}

baseline = pairwise_distances(ring)
for name, points in transformed.items():
    change = np.max(np.abs(pairwise_distances(points) - baseline))
    print(f'{name:26s} maximum distance change = {change:.3f}')

Explain why translation and rotation behave differently from anisotropic stretching in the distance calculation, even though all three continuous transformations are homeomorphisms here.

**▶ Likely sticking point.** Topological invariance of the underlying continuous shape does not make a fixed metric construction invariant to every homeomorphism.

## 3. Implement: choose and test a threshold

The next code cell samples a noisy ring and a filled disk. The two outputs are finite point clouds, not literal continuous spaces.

1. Predict which sample will tend to have an empty central region.
2. Inspect the plots.
3. Choose a threshold between `0.08` and `0.65` before seeing the sweep. Record why you chose it.

In [ ]:
n_noisy = 90
angles = RNG.uniform(0, 2 * np.pi, n_noisy)
noisy_ring = np.column_stack((np.cos(angles), np.sin(angles)))
noisy_ring += RNG.normal(0, 0.055, size=(n_noisy, 2))

disk_radii = np.sqrt(RNG.uniform(0, 1, n_noisy))
disk_angles = RNG.uniform(0, 2 * np.pi, n_noisy)
noisy_disk = np.column_stack((disk_radii * np.cos(disk_angles),
                              disk_radii * np.sin(disk_angles)))

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.7), constrained_layout=True)
for ax, (name, points) in zip(axes, [('noisy ring', noisy_ring),
                                     ('filled disk sample', noisy_disk)]):
    ax.scatter(points[:, 0], points[:, 1], s=18)
    ax.set(title=name, aspect='equal', xlim=(-1.2, 1.2), ylim=(-1.2, 1.2))
plt.show()

The supplied function joins pairs whose Euclidean distance is at most $\varepsilon$. It reports connected components and graph cycle rank

$$E-V+C.$$

In your notes, define $E$, $V$ and $C$. Predict what happens to both outputs as $\varepsilon$ increases, then run the sweep.

In [ ]:
epsilons = np.linspace(0.08, 0.65, 24)
results = {}
for name, points in [('noisy ring', noisy_ring), ('disk sample', noisy_disk)]:
    results[name] = [threshold_graph_statistics(points, epsilon)
                     for epsilon in epsilons]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.7), constrained_layout=True)
for name, records in results.items():
    axes[0].plot(epsilons, [r['components'] for r in records], label=name)
    axes[1].plot(epsilons, [r['graph_cycle_rank'] for r in records], label=name)
axes[0].set(xlabel=r'threshold $\varepsilon$', ylabel='connected components')
axes[1].set(xlabel=r'threshold $\varepsilon$', ylabel=r'graph cycle rank $E-V+C$')
for ax in axes:
    ax.legend()
plt.show()

chosen_epsilon = 0.28
for name, points in [('noisy ring', noisy_ring), ('disk sample', noisy_disk)]:
    print(name, threshold_graph_statistics(points, chosen_epsilon))

## 4. Compare: test your modelling choice

Use the next code cell to test three thresholds on the noisy ring and filled-disk samples. Include your original threshold choice, one smaller value and one larger value. Change only the entries in `student_epsilons`.

Then choose one extension:

- increase ring noise while keeping sample size and seed fixed;
- reduce sampling density; or
- anisotropically rescale both clouds.

State which single choice you changed and predict the direction of the effect.

In [ ]:
# TODO: replace these with your smaller, original and larger thresholds.
student_epsilons = [0.16, 0.28, 0.48]

for epsilon in student_epsilons:
    print(f'epsilon = {epsilon:.2f}')
    for name, points in [('noisy ring', noisy_ring), ('disk sample', noisy_disk)]:
        record = threshold_graph_statistics(points, epsilon)
        print(' ', name, record)

# OPTIONAL EXTENSION: make exactly one controlled change here.

## 5. Interpret: submit a short argument

Answer in complete sentences and refer to your output.

1. Why do matching scalar summaries not establish matching organisation?
2. Which quantities belong to the point-cloud representation, and which depend additionally on the graph construction?
3. At which of your thresholds did the conclusion change most, and why?
4. Why is graph cycle rank not yet $H_1$ of a Vietoris–Rips complex?
5. What additional evidence would be needed before making a claim about a dynamical-system attractor?

Finish with one sentence of the form: “This calculation supports ..., but it does not establish ...”.

**◇ Object check.** In a clique complex, every graph triangle is filled by a 2-simplex. Such a face can turn a graph cycle into a boundary. Topic 2 makes this algebra visible; Topic 3 constructs complexes across scale.